# D4 ★ 调通 API（第一个里程碑日）

> **今天结束时，你的程序要能自己跟互联网上的服务对话。**
>
> 这是整个计划最关键的一天。跨过这道门，后面的 RAG 和 Agent 全都是「调 API + 拼装」，
> 你就只是在重复今天学会的动作。

**今日产出：**

| 产出 | 说明 |
|---|---|
| `get_followers()` | 调 GitHub 公开 API（**不需要 Key**），查任意用户的粉丝数 |
| `.env` | 安全存放 API Key（**绝不写死在代码里**） |
| `chat()` | 调真实的 LLM API，问它一句话，拿到回复 |

**两个部分：**

- **第一部分（不需要注册）**：用 GitHub 公开 API 把「请求-响应」打通
- **第二部分（需要 Key）**：调真实的 LLM API，这才是后面项目要用的


---

## 先理解：API 到底是什么

**API = 你让别人的服务器帮你干活，它把结果用 JSON 还给你。**

一个生活类比：

| 点外卖 | 调 API |
|---|---|
| 打开外卖 App，找到那家店 | 拼出 URL（地址） |
| 选好菜，下单 | 把请求发出去（GET 或 POST） |
| 商家接单、做菜、出餐 | 对方的服务器处理 |
| 骑手把餐送到你手上 | 服务器把 **JSON 结果** 还给你 |
| 你打开包装，取出食物 | 你用 `.json()` 解析，取出要的字段 |

**关键差别**：点外卖你和人打交道，调 API 你和「一个地址」打交道。**你不需要知道对方怎么做的，只要知道往哪发、发什么、结果长什么样。**

昨天你已经练过「打开包装，取出食物」这一步了——就是 `json.loads` + 一层层剥字段。**今天只是把「从文件读 JSON」换成「从网络拿 JSON」。**


---

## 新语法：`requests.get()` —— 逐行表格

`requests` 是一个第三方库，专门负责发网络请求。它的用法非常简单。

**先看这四行：**

```
import requests

响应 = requests.get("https://api.github.com/users/octocat")
print(响应.status_code)
数据 = 响应.json()
```

| 行 | 代码 | 这句在干嘛 |
|---|---|---|
| 1 | `import requests` | 把 requests 这个工具箱拿进来。**和 `import json` 是一回事** |
| 2 | `requests.get(网址)` | 向那个地址发一个「取数据」的请求，等对方回复。回复被装进变量 `响应` |
| 3 | `响应.status_code` | 看对方回复的**状态码**：200 = 成功，404 = 找不到，401 = 没权限 |
| 4 | `响应.json()` | 把回复里的 JSON **文字**变成 Python 字典。**等价于昨天的 `json.loads(文本)`** |

**注意第 2 行和第 4 行的括号** —— 调用方法用圆括号，这是你昨天学过的。


In [3]:
import requests

# 第 2 行：向 GitHub 要 octocat 这个用户的信息
响应 = requests.get("https://api.github.com/users/octocat")

# 第 3 行：先看对方回复的状态码
print("状态码:", 响应.status_code)

# 第 4 行：把回复的 JSON 文字变成字典
数据 = 响应.json()

print("数据类型:", type(数据))
print()
print("随便看几个字段:")
print("   login       =", 数据["login"])
print("   name        =", 数据["name"])
print("   followers   =", 数据["followers"])
print("   public_repos=", 数据["public_repos"])

状态码: 200
数据类型: <class 'dict'>

随便看几个字段:
   login       = octocat
   name        = The Octocat
   followers   = 24586
   public_repos= 8


### 上面这四行，就是你今天要学的全部核心

对比一下昨天读文件的四行：

| 昨天（读文件） | 今天（调 API） |
|---|---|
| `with open("sample/api_response.json") as 文件:` | `响应 = requests.get(网址)` |
| `JSON文字 = 文件.read()` | （requests 已经帮你读好了，在 `响应` 里） |
| `数据 = json.loads(JSON文字)` | `数据 = 响应.json()` |
| `数据["choices"][0]["message"]["content"]` | **一模一样，一个字都不用改** |

**这不是新知识，这是旧知识换了个数据来源。**


---

## 新语法：状态码 —— 必须先检查这个

对方的服务器不会永远成功。**状态码就是对方给你的「回复类型」。**

| 状态码 | 意思 | 你要做什么 |
|---|---|---|
| **200** | 成功 | 正常取数据 |
| **404** | 找不到（这个用户/网址不存在） | 返回「没找到」，别崩 |
| **401 / 403** | 没权限（Key 不对或没带 Key） | 检查 Key |
| **500+** | 对方服务器自己出问题了 | 等一会儿重试 |

**为什么必须先检查？** 因为你如果对着一个 404 的回复去取 `数据["followers"]`，程序会直接崩——因为 404 回复里根本没有 `followers` 这个字段，它只有 `{"message": "Not Found"}`。

看下面这个例子：

In [4]:
# 故意访问一个不存在的用户
响应404 = requests.get("https://api.github.com/users/no-such-user-xyz-9527")

print("状态码:", 响应404.status_code)
print("对方回了什么:", 响应404.json())
print()

# 如果不管状态码，直接去取字段会怎样？
try:
    print(响应404.json()["followers"])
except KeyError as e:
    print(f"❌ 直接取字段 → KeyError: {e}")
    print("   因为 404 的回复里根本没有 followers 这个键")
    print("   这个 KeyError，就是你昨天学的「字典里没有这个键」")

状态码: 404
对方回了什么: {'message': 'Not Found', 'documentation_url': 'https://docs.github.com/rest', 'status': '404'}

❌ 直接取字段 → KeyError: 'followers'
   因为 404 的回复里根本没有 followers 这个键
   这个 KeyError，就是你昨天学的「字典里没有这个键」


---

## `响应.json()` 和 `json.loads()` 是什么关系

**它们是一回事。** 看证据：

In [5]:
import json

# 方式一：先拿到文字，再用 json.loads 解析（昨天学的）
文字 = 响应.text                       # .text 是「对方回的原始文字」
数据A = json.loads(文字)

# 方式二：直接用 .json()（今天学的）
数据B = 响应.json()

print("两种方式拿到的结果一样吗?", 数据A == 数据B)
print()
print("所以：响应.json()  就是  json.loads(响应.text)  的简写")
print()
print("原始文字长这样（前 100 个字）:")
print(文字[:100])

两种方式拿到的结果一样吗? True

所以：响应.json()  就是  json.loads(响应.text)  的简写

原始文字长这样（前 100 个字）:
{"login":"octocat","id":583231,"node_id":"MDQ6VXNlcjU4MzIzMQ==","avatar_url":"https://avatars.github


---

## 任务 D4-1：写一个查粉丝数的函数

**要求**：传入 GitHub 用户名，返回这个用户的粉丝数。**查不到就返回 `None`，不要崩。**

**骨架：**

```
def get_followers(username):
    网址 = f"https://api.github.com/users/{username}"    # ① 把用户名拼进网址
    响应 = requests.get(网址)                            # ② 发请求

    if 响应.status_code != 200:                          # ③ 状态码不是 200
        return None                                      #    直接返回 None

    数据 = 响应.json()                                    # ④ 解析
    return 数据["followers"]                              # ⑤ 取出粉丝数
```

**这五步就是所有 API 调用的标准骨架。** 你后面写 RAG、写 Agent，每一次调 API 都是这五步，只是网址、参数、要取的字段不同。

**想一想第 ③ 步为什么必须写**：想想上面那个 404 的例子。

In [12]:
def get_followers(username):
    网址 = f"https://api.github.com/users/{username}"
    响应 = requests.get(网址)

    # 想一想：状态码不是 200 的时候怎么办？
    if 响应.status_code != 200:
        return None
    
    # 提示：return None

    数据 = 响应.json()
    return 数据["followers"]


# 自测（数字来自真实请求，可能会缓慢变化，所以用大小比较）
assert get_followers("octocat") > 20000, "octocat 的粉丝数应该很多"
assert get_followers("torvalds") > 100000, "torvalds 的粉丝数应该更多"
assert get_followers("no-such-user-xyz-9527") is None, "不存在的用户应该返回 None"
print("D4-1 通过 ✅")

D4-1 通过 ✅


---

# 第二部分：调真实的 LLM API

**这部分才是后面 RAG / Agent 项目真正要用的东西。**

和 GitHub API 的区别只有一个：**LLM API 需要钥匙（API Key）**。

| | GitHub 公开 API | LLM API |
|---|---|---|
| 要不要 Key | 不要 | **必须要** |
| 请求方式 | GET（取数据） | POST（提交数据，因为你要把问题发过去） |
| 花钱吗 | 免费 | 按用量收费（很便宜，几分钱） |

## 第一步（你要自己做）：拿一个 API Key

**这一步只能你本人做**——需要注册账号，我不会向你索要任何密钥。

1. 打开 **platform.deepseek.com**，注册并登录（手机号即可）
2. 左侧找到 **API Keys** → 点 **创建 API Key**
3. **立刻复制那串 Key**（形如 `sk-xxxxxxxx`）—— 页面关掉后就再也看不到完整内容了
4. 新账号一般有免费额度，够你做完整个计划


## 第二步：把 Key 放进 `.env` 文件（安全规则）

**绝对不要把 Key 写死在代码里。** 原因：

- 代码要提交到 GitHub，Key 会一起泄露
- 别人拿到你的 Key 可以花你的钱
- 面试官看到代码里硬编码 Key，会直接扣分

**正确做法**：Key 放在一个叫 `.env` 的文件里，代码去读它，而 `.env` 永远不提交。

**你要创建两个文件：**

**文件 1：`.env`（真实 Key，不进仓库）**

```
DEEPSEEK_API_KEY=sk-你复制的那串
```

**文件 2：`.env.example`（模板，可以进仓库，给别人看格式）**

```
DEEPSEEK_API_KEY=在这里填你自己的Key
```

好消息：`.gitignore` 里已经写了 `.env`，**你的 Key 不会被提交上去**。我待会儿给你建好 `.env.example`。


## 新语法：POST 请求 —— 逐行表格

调 LLM 用的是 **POST**（因为你是在「提交」一个提问），比 GET 多了两样东西：**请求头（headers）** 和 **请求体（body）**。

```
import requests

网址 = "https://api.deepseek.com/chat/completions"

请求头 = {
    "Authorization": f"Bearer {key}",
    "Content-Type": "application/json",
}

提问内容 = {
    "model": "deepseek-chat",
    "messages": [{"role": "user", "content": "用一句话解释什么是 RAG"}],
}

响应 = requests.post(网址, headers=请求头, json=提问内容)
数据 = 响应.json()
回复 = 数据["choices"][0]["message"]["content"]
```

| 行 | 代码 | 这句在干嘛 |
|---|---|---|
| 1 | `网址 = "https://api.deepseek.com/chat/completions"` | LLM 服务的地址 |
| 2 | `"Authorization": f"Bearer {key}"` | **通行证**：告诉对方「我是谁」。`Bearer` 是固定前缀，后面接你的 Key |
| 3 | `"Content-Type": "application/json"` | **声明格式**：告诉对方「我发给你的是 JSON」 |
| 4 | `"model": "deepseek-chat"` | 用哪个模型（不同模型价格不同） |
| 5 | `"messages": [{"role": "user", "content": "..."}]` | **对话内容**。这是个列表，每个元素是一条消息 |
| 6 | `requests.post(网址, headers=..., json=...)` | 把三样东西一起发出去：地址、通行证、内容 |
| 7 | `数据["choices"][0]["message"]["content"]` | **和昨天练的一模一样** —— 一层层剥出回复文字 |

**第 7 行你已经会了。** 昨天在 `sample/api_response.json` 上练的就是这个，DeepSeek 返回的结构几乎一样。


In [1]:
# ============================================================
# 下面是完整模板。把 Key 填进 .env 之后，取消注释就能跑
# ============================================================
# import os
# from dotenv import load_dotenv
#
# load_dotenv()                                  # 把 .env 里的内容读进环境变量
# key = os.getenv("DEEPSEEK_API_KEY")            # 按名字取出 Key
#
# if not key:
#     print("❌ 没找到 DEEPSEEK_API_KEY，检查 .env 文件建好了吗")
# else:
#     print("✅ Key 读取成功，前 8 位是:", key[:8] + "...")


# ---------- 检查 .env 是否配好（这一段可以直接运行）----------
import os
from dotenv import load_dotenv

load_dotenv()
key = os.getenv("DEEPSEEK_API_KEY")

if not key or key.startswith("在这里"):
    print("⬜ 还没配好 Key —— 按上面的步骤建 .env 文件，然后重新运行这一格")
else:
    print("✅ Key 读取成功，前 8 位:")
    print("   ", key[:8] + "..." + key[-4:])
    print("   （只显示头尾，避免泄露）")

✅ Key 读取成功，前 8 位:
    sk-8fbe1...8f17
   （只显示头尾，避免泄露）


## 任务 D4-2：写一个 `chat()` 函数

**要求**：传入一个问题，返回模型的回复文字。

先把上面模板里注释掉的代码**取消注释**，然后按注释填完函数体。

**注意第一行的 `load_dotenv()`** —— 它必须在 `os.getenv` **之前**执行，否则读不到 Key。这是个执行顺序问题，和你昨天踩的 Jupyter 顺序坑是同一类。

In [12]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()


def chat(问题):
    网址 = "https://api.deepseek.com/chat/completions"

    请求头 = {
        "Authorization": f"Bearer {os.getenv('DEEPSEEK_API_KEY')}",
        "Content-Type": "application/json",
    }

    提问内容 = {
        "model": "deepseek-chat",
        "messages": [{"role": "user", "content": 问题}],
    }

    响应 = requests.post(网址, headers=请求头, json=提问内容)
    if 响应.status_code != 200:
        return None
    数据 = 响应.json()
    return 数据["choices"][0]["message"]["content"]

    # ① 想一想：状态码不是 200 的时候怎么办？
    #    提示：打印状态码和 响应.json()，方便排查，然后 return None

    # ② 想一想：怎么剥出回复文字？
    #    提示：和 D4-1 一样两步 —— 先 数据 = 响应.json()
    #          然后 return 数据["choices"][0]["message"]["content"]


print(chat("AI大数据找工作方案"))

# AI大数据找工作方案

以下是利用AI和大数据技术提升求职效率的完整方案：


## 一、整体思路

传统求职 vs AI大数据求职：

| 环节 | 传统方式 | AI大数据方式 |
|------|----------|--------------|
| 简历 | 一份通用简历海投 | AI根据JD自动定制简历 |
| 找岗位 | 手动刷招聘网站 | 爬虫+算法聚合匹配 |
| 匹配 | 凭感觉判断 | 语义匹配度打分 |
| 面试 | 自己摸索 | AI模拟面试+题库预测 |
| 谈薪 | 信息不对称 | 大数据薪资对标 |


## 二、具体操作方案

### 1. 岗位数据聚合

**数据来源：**
- BOSS直聘、猎聘、拉勾、智联
- 脉脉、LinkedIn
- 公司官网招聘页
- 国企/事业单位招聘公告

**技术手段：**
- Python爬虫（Scrapy/Selenium）定时抓取
- RSS订阅 + API接口
- 关键词监控（如“Python+3年+北京”）

### 2. 简历智能优化

- 用GPT/Claude等大模型，把JD和你的经历一起输入，让它生成匹配版简历
- 关键词提取：从JD中提取高频技能词，确保简历覆盖
- ATS优化：很多公司用机器筛简历，需优化格式和关键词密度
- 量化成果：AI帮你把“负责XX”改写成“通过XX使XX提升XX%”

### 3. 岗位匹配打分

```
匹配度 = 技能匹配(40%) + 经验匹配(25%) + 薪资匹配(15%) + 地点/行业(20%)
```

用AI对每个岗位JD做语义分析，输出匹配分数，优先投递>80分的岗位。

### 4. 薪资大数据对标

- 数据源：看准网、职友集、脉脉、OfferShow
- 按城市/行业/岗位/年限做分位数分析（P25/P50/P75）
- 谈判时用数据说话，避免被压价

### 5. AI模拟面试

- 用大模型扮演面试官，基于目标岗位JD出题
- 常见工具：ChatGPT语音模式、豆包、Kimi
- 行为面试：STAR法则训练
- 技术面试：LeetCode + AI讲解

### 6. 投递策略优化

- 黄金时间：工作日 9:00-11:00 投递回复率最高
- 优先级排序：匹配度高的先投
- 跟进节奏：投递后3天无回复可主动打招呼


---

## 卡住了怎么办

| 报错 | 什么意思 | 怎么办 |
|---|---|---|
| `ModuleNotFoundError: No module named 'requests'` | 库没装 | 你已经装好了，不会遇到 |
| `401` 或 `403` | Key 不对 / 没带 Key | 检查 `.env` 里 Key 有没有复制全（`sk-` 开头那一整串） |
| `402` | 余额不足 | 去 DeepSeek 后台看额度 |
| `KeyError: 'choices'` | 回复里没有 choices | 说明请求失败了，**先把 `响应.json()` 打印出来看** |
| `NameError: name 'os' is not defined` | 忘了 `import os` | 检查开头有没有 import |
| Key 读出来是 None | `.env` 没建好 / 位置不对 | `.env` 必须和 notebook 在**同一个目录**（~/ai-d1/） |

**万能排查法：把 `响应.json()` 整个打印出来。**

```
print(响应.status_code)
print(响应.json())
```

**看得见的错误，永远比瞎猜快。** 这是你昨天学到的调试方法，今天同样适用。

---

## 收尾

**今天的验收标准：**

1. `get_followers("octocat")` 能返回一个很大的数字
2. `get_followers("随便乱敲的名字")` 返回 `None` 而不是崩掉
3. `chat("你好")` 能打印出模型的回复
4. 你的 Key 在 `.env` 里，`git status` 看不到它

**做完这四条，你今天就是在原计划的 10/16 之前，提前完成了第一个里程碑。**
